In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
            
    main_features.append(f'ME_prod_M*_E*')
    main_features.append(f'ME_ratio_M*_E*')
            
    # M × I (Market × Interest Rate)
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    
    main_features.append(f'MI_prod_M*_I*')
    main_features.append(f'MI_ratio_M*_I*')
    main_features.append(f'MI_spread_M*_I*')
        
    # M × P (Market × Price)
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    
    main_features.append(f'MP_prod_M*_P*')
    main_features.append(f'MP_ratio_M*_P*')
    
    # M × V (Market × Volatility)
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
    # Sharpe-like ratio
    if 'return' in 'M*'.lower():
        data[f'MV_sharpe_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
        main_features.append(f'MV_sharpe_M*_V*')
                
    main_features.append(f'MV_ratio_M*_V*')
    main_features.append(f'MV_prod_M*_V*')
        
    # M × S (Market × Sentiment)
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.append(f'MS_prod_M*_S*')
    main_features.append(f'MS_weighted_M*_S*')
        
    # E × I (Economic × Interest Rate)
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.append(f'EI_diff_E*_I*')
    main_features.append(f'EI_ratio_E*_I*')
    main_features.append(f'EI_prod_E*_I*')
        
    # E × P (Economic × Price)
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.append(f'EP_prod_E*_P*')
    main_features.append(f'EP_ratio_E*_P*')
        
    # E × V (Economic × Volatility)
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.append(f'EV_prod_E*_V*')
    main_features.append(f'EV_uncertainty_E*_V*')
        
    # E × S (Economic × Sentiment)
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.append(f'ES_prod_E*_S*')
    main_features.append(f'ES_diff_E*_S*')
        
    # I × P (Interest Rate × Price)
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.append(f'IP_prod_I*_P*')
    main_features.append(f'IP_discount_I*_P*')
    
    # I × V (Interest Rate × Volatility)
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
        
    # I × S (Interest Rate × Sentiment)
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
        
    # P × V (Price × Volatility)
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.append(f'PV_prod_P*_V*')
    main_features.append(f'PV_stability_P*_V*')
        
    # P × S (Price × Sentiment)
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    # Contrarian signal (opposite signs amplify)
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.append(f'PS_prod_P*_S*')
    main_features.append(f'PS_contrarian_P*_S*')
        
    # V × S (Volatility × Sentiment)
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.append(f'VS_prod_V*_S*')
    main_features.append(f'VS_panic_V*_S*')

    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 

    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 

    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 

    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']

    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()

    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    return data

train_processed = preprocessing(train, "train")

train_split, val_split = train_test_split(
    train_processed, test_size=0.1, random_state=42
)

train_x = train_split.drop(columns=["target"])
test_x = val_split.drop(columns=["target"])
train_y = train_split['target']
test_y = val_split['target']

In [3]:
train_x

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,IP_prod_I*_P*,IP_discount_I*_P*,IV_prod_I*_V*,IS_prod_I*_S*,PV_prod_P*_V*,PV_stability_P*_V*,PS_prod_P*_S*,PS_contrarian_P*_S*,VS_prod_V*_S*,VS_panic_V*_S*
8732,2.451942,0.119378,0.000661,0.000661,0.000661,0.000661,0.857804,-0.701547,-1.125624,-0.430446,...,35.385341,0.728472,-0.096819,44.320302,-0.081402,-365.478565,37.262803,-37.262803,-0.101956,-0.101956
7258,0.972619,0.372024,0.001984,0.001984,0.001984,0.001984,0.101852,0.904259,1.593782,0.909865,...,38.069255,0.404987,66.950369,-13.322198,30.058526,0.568619,-5.981231,5.981231,-10.518872,10.518872
8637,3.460797,0.297950,0.002976,0.002976,0.002976,0.002976,0.944775,-0.335688,-0.598544,-0.321239,...,32.610638,0.481894,23.928640,46.123336,13.020600,1.362829,25.097686,-25.097686,18.415877,18.415877
7503,2.173250,0.005952,0.003307,0.003307,0.003307,0.003307,0.030423,-2.044277,-2.050591,-0.546969,...,10.490228,0.775896,16.762176,4.365041,17.056287,0.625827,4.441630,-4.441630,7.097213,7.097213
7775,1.459825,0.141534,0.004630,0.004630,0.004630,0.004630,0.096561,1.620175,1.302591,0.856185,...,-2.706615,-20.889961,-7.080123,-4.182411,15.384020,0.382284,9.087736,-9.087736,23.772238,23.772238
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8099,1.345860,0.773810,0.006944,0.006944,0.006944,0.006944,0.441468,-0.028077,0.449144,0.903182,...,6.062446,2.459914,12.553895,1.957006,57.791777,0.482914,9.009063,-9.009063,18.655644,18.655644
8263,1.382723,0.834656,0.015873,0.002646,0.002646,0.002646,0.461310,-1.329866,-1.088179,-0.964261,...,24.643086,0.169677,79.113748,-36.463646,14.584848,0.311489,-6.722178,6.722178,-21.580768,21.580768
7829,1.347717,0.123677,0.051587,0.019841,0.019841,0.006614,0.432870,1.488317,1.584364,1.726731,...,-6.650427,-2.389578,-31.258203,-17.745646,41.371909,0.212758,23.487315,-23.487315,110.394607,110.394607
8428,1.472010,0.515212,0.003638,0.003638,0.029431,0.003638,0.715939,-1.571494,-1.598439,-1.271266,...,42.267519,0.439360,11.290088,69.695704,5.492587,3.743772,33.906709,-33.906709,9.056830,9.056830


In [4]:
train_y

8732   -0.020579
7258   -0.004489
8637   -0.005918
7503   -0.001114
7775   -0.001467
          ...   
8099    0.015049
8263   -0.007086
7829   -0.020020
8428   -0.003406
8095   -0.021361
Name: target, Length: 1818, dtype: float64

In [5]:
test_x

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,IP_prod_I*_P*,IP_discount_I*_P*,IV_prod_I*_V*,IS_prod_I*_S*,PV_prod_P*_V*,PV_stability_P*_V*,PS_prod_P*_S*,PS_contrarian_P*_S*,VS_prod_V*_S*,VS_panic_V*_S*
7643,2.137565,0.082672,0.005952,0.005952,0.018519,0.005952,0.066138,-5.036853,-3.315876,-1.620565,...,0.010089,-0.109937,-1.542803,0.035555,-1.489733,-0.006539,0.034332,-0.034332,-5.250235,5.250235
8352,1.620099,0.639881,0.013889,0.001323,0.014550,0.001323,0.644841,-1.735948,-1.704098,-1.335877,...,45.456193,0.363007,6.253054,30.592774,2.482012,7.269439,12.143130,-12.143130,1.670436,1.670436
7689,1.939656,0.128638,0.003638,0.003638,0.003638,0.003638,0.060516,0.095427,-0.282266,-0.273780,...,-0.701169,6.117274,-10.118814,-4.539251,9.418332,0.069294,4.225018,-4.225018,60.972735,60.972735
7559,2.142929,0.379299,0.006614,0.006614,0.006614,0.006614,0.086310,-1.870163,-1.766520,-0.762412,...,11.389905,1.042318,4.938644,21.160280,6.958097,2.306282,29.812893,-29.812893,12.926823,12.926823
7545,2.142403,0.383929,0.018519,0.003968,0.003968,0.003968,0.003307,-1.919986,-1.811433,-0.771263,...,10.742726,1.165411,9.448990,17.293525,15.285114,1.136918,27.974791,-27.974791,24.605813,24.605813
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8806,2.134607,0.094907,0.027447,0.014881,0.002976,0.002976,0.881944,-0.273881,-0.733726,-0.545524,...,36.711508,0.938923,28.377193,22.737325,31.259350,1.293698,25.046664,-25.046664,19.360523,19.360523
8980,1.577651,0.186177,0.001323,0.001323,0.001323,0.001323,0.955026,-0.583419,-0.704264,0.298365,...,25.556747,1.008152,16.689441,16.548431,20.515576,1.531312,20.342240,-20.342240,13.284187,13.284187
8363,1.596276,0.636243,0.016865,0.004299,0.004299,0.004299,0.648479,-1.191415,-1.612416,-0.872927,...,40.762763,0.339060,29.851665,6.164704,11.087660,1.365511,2.289726,-2.289726,1.676828,1.676828
8297,1.326434,0.678902,0.005622,0.005622,0.005622,0.005622,0.516204,-1.290387,-1.482997,-1.545220,...,40.470395,0.304275,29.767939,57.151235,9.877819,1.359530,18.964347,-18.964347,13.949197,13.949197


In [6]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
        
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_mape = mean_absolute_percentage_error(train_y, current_train_pred)
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
            
            if test_x is not None and test_y is not None:
                current_test_pred = sum(m.predict(test_x) for m in self.models)
                test_mape = mean_absolute_percentage_error(test_y, current_test_pred)
                print(f'Cumulative Test MAPE: {test_mape:.4f}')
            
            print()
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}


LGBM_R_parm = {
    'boosting_type': 'gbdt', 
    'colsample_bytree': 0.9484106149593443, 
    'learning_rate': 0.1988123373955639, 
    'max_bin': 77, 
    'max_depth': 10, 
    'metric': 'mape', 
    'min_child_samples': 81, 
    'min_data_in_leaf': 21, 
    'n_estimators': 5029, 
    'num_leaves': 42, 
    'objective': 'regression_l1', 
    'reg_alpha': 0.6355835028602363, 
    'reg_lambda': 3.109823217156622, 
    'subsample': 0.7300733288106989, 
    'verbosity': -1
}

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y, test_x, test_y)

ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

print('\nFinal Test Evaluation:')
ensemble.evaluate(test_x, test_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAPE: 136714476.1298
Cumulative Test MAPE: 2.8377

Training Model 2...
Cumulative Training MAPE: 50301150.7166
Cumulative Test MAPE: 2.8532

Training Model 3...
Cumulative Training MAPE: 529767124.3095
Cumulative Test MAPE: 2.8686

Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
MAPE: 529767124.3095
MAE: 0.0015
RMSE: 0.0041

Final Test Evaluation:
MAPE: 2.8686
MAE: 0.0079
RMSE: 0.0109


In [7]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2

class ParticipantVisibleError(Exception):
    pass

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = None) -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')

    solution = solution.copy()
    solution['position'] = submission['prediction'].values

    if solution['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if solution['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].min()} below minimum of {MIN_INVESTMENT}')

    solution['strategy_returns'] = solution['risk_free_rate'] * (1 - solution['position']) + solution['position'] * solution['forward_returns']

    strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
    strategy_std = solution['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
    market_std = solution['forward_returns'].std()

    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')

    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    return_gap = max(
        0,
        (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr,
    )
    return_penalty = 1 + (return_gap**2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


print("\nEvaluating on validation set...")
val_predictions = LGBM_R_model.predict(test_x)
val_signals = np.array([convert_ret_to_signal(pred) for pred in val_predictions])

val_indices = val_split.index
val_original = train.loc[val_indices].copy()

solution_df = pd.DataFrame({
    'forward_returns': val_original['forward_returns'].values,
    'risk_free_rate': val_original['risk_free_rate'].values
})

submission_df = pd.DataFrame({
    'prediction': val_signals
})

try:
    validation_score = score(solution_df, submission_df)
    print(f"Validation Adjusted Sharpe Ratio: {validation_score}")
except Exception as e:
    print(f"Error calculating validation score: {e}")


Evaluating on validation set...
Validation Adjusted Sharpe Ratio: 0.7183362427235841


In [8]:
def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        
        test_processed = preprocessing(test_df, 'inference')
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))